# Phase 4: SQLite Database Development

## Load Cleaned Datasets

In [1]:
import pandas as pd

telco = pd.read_csv(
    "../data/processed/clean_telco.csv"
)

retail_churn = pd.read_csv(
    "../data/processed/clean_retail_churn.csv"
)

retail_sales = pd.read_csv(
    "../data/processed/clean_retail_sales.csv"
)

## Create SQLite Database

In [3]:
import sqlite3

conn = sqlite3.connect(
    "../sqlite/churn_analysis.db"
)

cursor = conn.cursor()

# verify

cursor.execute(
    "SELECT sqlite_version();"
)

print(cursor.fetchone())

('3.53.4',)


## Design Database Schema

### Telco Tables

#### customers_telco

Customer information:
- CustomerID
- Gender
- Senior Citizen
- Partner
- Dependents
- Married
- Age
- City
- State

#### services_telco
- CustomerID
- Phone Service
- Internet Service
- Online Security
- Online Backup
- Device Protection
- Tech Support
- Streaming TV
- Streaming Movies

#### billing_telco
- CustomerID
- Tenure Months
- Monthly Charges
- Total Charges
- Contract
- Payment Method
- Paperless Billing

#### churn_telco
- CustomerID
- Churn Label
- Churn Value
- Churn Score
- CLTV
- Churn Reason

### Retail Churn Tables

#### retail_customers
- CustomerID
- Gender
- MaritalStatus
- CityTier
- PreferredLoginDevice
- PreferredPaymentMode

#### retail_behavior
- CustomerID
- Tenure
- HourSpendOnApp
- CouponUsed
- OrderCount
- DaySinceLastOrder
- CashbackAmount

#### retail_churn_status
- CustomerID
- Churn
- Complain
- SatisfactionScore

### Retail Sales Tables

#### retail_transactions
- customer_id
- transaction_id
- transaction_date
- product_id
- quantity
- unit_price
- total_sales

#### retail_customers_sales
- customer_id
- age
- income_bracket
- membership_years
- loyalty_program
- churned

## Load Data into SQLite

In [5]:
telco.to_sql(
    "telco_customers",
    conn,
    if_exists="replace",
    index=False
)

retail_churn.to_sql(
    "retail_churn_customers",
    conn,
    if_exists="replace",
    index=False
)

retail_sales.to_sql(
    "retail_sales_customers",
    conn,
    if_exists="replace",
    index=False
)

1000000

## Verify Tables

In [6]:
pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn
)

,name
0,telco_customers
1,retail_churn_customers
2,retail_sales_customers


## Create SQL Analytics Queries

### Telco Analytics

#### Query 1: Telco Churn Rate

In [10]:
query = "SELECT [Churn Label], COUNT(*) AS CustomerCount FROM telco_customers GROUP BY [Churn Label]"

pd.read_sql(query, conn)

,Churn Label,CustomerCount
0,No,5174
1,Yes,1869


#### Query 2: Churn by Contract

In [13]:
query = "SELECT Contract, [Churn Label], COUNT(*) AS CustomerCount FROM telco_customers GROUP BY Contract, [Churn Label] ORDER BY CustomerCount DESC"
pd.read_sql(query, conn)
# Which contract types experience the most churn?

,Contract,Churn Label,CustomerCount
0,Month-to-month,No,2220
1,Month-to-month,Yes,1655
2,Two year,No,1647
3,One year,No,1307
4,One year,Yes,166
5,Two year,Yes,48


#### Query 3: Average Charges by Churn

In [14]:
query = "SELECT [Churn Label], AVG([Monthly Charges]) AS AvgMonthlyCharge FROM telco_customers GROUP BY [Churn Label]"
pd.read_sql(query, conn)
# Do churned customers pay higher monthly charges?

,Churn Label,AvgMonthlyCharge
0,No,61.265124
1,Yes,74.441332


### Retail Churn Analytics

#### Query 4: Churn by Preferred Payment Mode

In [15]:
query = "SELECT PreferredPaymentMode, Churn, COUNT(*) AS Customers FROM retail_churn_customers GROUP BY PreferredPaymentMode, Churn ORDER BY Customers DESC"
pd.read_sql(query, conn)

,PreferredPaymentMode,Churn,Customers
0,Debit Card,0,1958
1,Credit Card,0,1308
2,E Wallet,0,474
3,Debit Card,1,356
4,Upi,0,342
5,Cod,0,260
6,Cc,0,214
7,Credit Card,1,193
8,E Wallet,1,140
9,Cash On Delivery,0,126


#### Query 5: Complaints vs Churn

In [16]:
query = "SELECT Complain, Churn, COUNT(*) AS Customers FROM retail_churn_customers GROUP BY Complain, Churn"
pd.read_sql(query, conn)
# Are complaining customers more likely to churn?

,Complain,Churn,Customers
0,0,0,3586
1,0,1,440
2,1,0,1096
3,1,1,508


#### Query 6: Satisfaction Score by Churn

In [17]:
query = "SELECT Churn, AVG(SatisfactionScore) AS AvgSatisfaction FROM retail_churn_customers GROUP BY Churn"
pd.read_sql(query, conn)

,Churn,AvgSatisfaction
0,0,3.001282
1,1,3.390295


### Retail Sales Analytics

#### Query 7: Churn by Loyalty Program

In [18]:
query = "SELECT loyalty_program, churned, COUNT(*) AS CustomerCount FROM retail_sales_customers GROUP BY loyalty_program, churned"
pd.read_sql(query, conn)

,loyalty_program,churned,CustomerCount
0,No,No,250208
1,No,Yes,250080
2,Yes,No,250063
3,Yes,Yes,249649


#### Query 8: Average Total Sales by Churn

In [19]:
query = "SELECT churned, AVG(total_sales) AS AvgSales FROM retail_sales_customers GROUP BY churned"
pd.read_sql(query, conn)

,churned,AvgSales
0,No,5057.651372
1,Yes,5054.466432


#### Query 9: Top Product Categories

In [20]:
query = "SELECT product_category, SUM(total_sales) AS Revenue FROM retail_sales_customers GROUP BY product_category ORDER BY Revenue DESC LIMIT 10"
pd.read_sql(query, conn)

,product_category,Revenue
0,Toys,1.013781e+09
1,Clothing,1.011086e+09
2,Groceries,1.011073e+09
3,Electronics,1.010576e+09
4,Furniture,1.009544e+09


### Create SQL Results Report

In [24]:
from pathlib import Path
import pandas as pd

report_path = Path(
    "../reports/sqlite_analysis_summary.md"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write("# SQLite Analysis Summary\n\n")

    f.write(
        "This report summarizes the SQL analysis performed on the cleaned customer churn datasets.\n\n"
    )

    # ==================================================
    # TELCO ANALYSIS
    # ==================================================

    f.write("## Telco Customer Churn Analysis\n\n")

    queries = {
        "Churn Distribution": "SELECT [Churn Label], COUNT(*) AS CustomerCount FROM telco_customers GROUP BY [Churn Label]",

        "Contract vs Churn": "SELECT Contract, [Churn Label], COUNT(*) AS CustomerCount FROM telco_customers GROUP BY Contract, [Churn Label] ORDER BY CustomerCount DESC",

        "Average Monthly Charges by Churn": "SELECT [Churn Label], ROUND(AVG([Monthly Charges]),2) AS AvgMonthlyCharge FROM telco_customers GROUP BY [Churn Label]"
    }

    for title, query in queries.items():

        f.write(f"### {title}\n\n")

        result = pd.read_sql(
            query,
            conn
        )

        f.write(
           result.to_markdown(index=False)
        )

        top_row = result.iloc[0]

        f.write(
            f"\nObservation: Highest value is {top_row.to_dict()}\n\n"
        )

        f.write("\n\n")

    # ==================================================
    # RETAIL CHURN ANALYSIS
    # ==================================================

    f.write(
        "## Retail Customer Churn Analysis\n\n"
    )

    retail_queries = {

        "Preferred Payment Mode vs Churn": "SELECT PreferredPaymentMode, Churn, COUNT(*) AS Customers FROM retail_churn_customers GROUP BY PreferredPaymentMode, Churn ORDER BY Customers DESC",

        "Complaints vs Churn": "SELECT Complain, Churn, COUNT(*) AS Customers FROM retail_churn_customers GROUP BY Complain, Churn",

        "Average Satisfaction Score": "SELECT Churn, ROUND( AVG(SatisfactionScore),2 ) AS AvgSatisfaction FROM retail_churn_customers GROUP BY Churn"
    }

    for title, query in retail_queries.items():

        f.write(f"### {title}\n\n")

        result = pd.read_sql(
            query,
            conn
        )

        f.write(
            result.to_markdown(index=False)
        )

        top_row = result.iloc[0]

        f.write(
            f"\nObservation: Highest value is {top_row.to_dict()}\n\n"
        )

        f.write("\n\n")

    # ==================================================
    # RETAIL SALES ANALYSIS
    # ==================================================

    f.write(
        "## Retail Sales Analysis\n\n"
    )

    sales_queries = {

        "Loyalty Program vs Churn": "SELECT loyalty_program, churned, COUNT(*) AS CustomerCount FROM retail_sales_customers GROUP BY loyalty_program, churned",

        "Average Total Sales by Churn": "SELECT churned, ROUND( AVG(total_sales),2 ) AS AvgSales FROM retail_sales_customers GROUP BY churned",

        "Top Product Categories": "SELECT product_category, ROUND( SUM(total_sales),2 ) AS Revenue FROM retail_sales_customers GROUP BY product_category ORDER BY Revenue DESC LIMIT 10"
    }

    for title, query in sales_queries.items():

        f.write(f"### {title}\n\n")

        result = pd.read_sql(
            query,
            conn
        )

        f.write(
            result.to_markdown(
                index=False
            )
        )

        top_row = result.iloc[0]

        f.write(
            f"\nObservation: Highest value is {top_row.to_dict()}\n\n"
        )

        f.write("\n\n")

print(
    f"Created report: {report_path}"
)

Created report: ..\reports\sqlite_analysis_summary.md


### Database Validation

In [21]:
for table in [
    "telco_customers",
    "retail_churn_customers",
    "retail_sales_customers"
]:
    
    query = f"SELECT COUNT(*) FROM {table}"
    
    print(
        table,
        pd.read_sql(query, conn)
              .iloc[0,0]
    )

telco_customers 7043
retail_churn_customers 5630
retail_sales_customers 1000000
